In [1]:
import numpy as np
import pandas as pd
import random
import os

# Read observed data

In [2]:
# Get the name of all the files in the output folder
folder_path = "/raid/sepideh/Project_MCL/1D-AEMpy-UW-metabolism-BM/output"
files = [f for f in os.listdir(folder_path) if 'obs' in f]

files

['do_obs.csv',
 'doc_obs.csv',
 'fdom_obs.csv',
 'phyco_obs.csv',
 'poc_obs.csv',
 'secchi_obs.csv',
 'temp_obs.csv',
 'scaled_chla_obs.csv']

In [3]:
# Read the files and store them in a dictionary.
# Access the datasets: df["file name"]
df_melt = {}
for f in files:
    df = pd.read_csv(os.path.join(folder_path, f))
    df_melt[f] = pd.melt(df, id_vars=["datetime"], var_name="depth", value_name=os.path.splitext(f)[0])


In [4]:
df_melt.keys()

dict_keys(['do_obs.csv', 'doc_obs.csv', 'fdom_obs.csv', 'phyco_obs.csv', 'poc_obs.csv', 'secchi_obs.csv', 'temp_obs.csv', 'scaled_chla_obs.csv'])

## Get the main timeline and depths

In [6]:
folder_path = '/raid/sepideh/Project_MCL/1D-AEMpy-UW-metabolism-BM'
file_path = os.path.join(folder_path, "all_data_lake_modeling_process_based.csv")

df_main = pd.read_csv(file_path)

In [7]:
depth_steps = 50
valid_depths = list(np.array(range(0, depth_steps)).astype(float))

df_observed = df_main[["datetime", "depth"]].reset_index(drop=True)
df_observed

,datetime,depth
0,2016-05-17 18:00:00,0.0
1,2016-05-17 18:00:00,1.0
2,2016-05-17 18:00:00,2.0
3,2016-05-17 18:00:00,3.0
4,2016-05-17 18:00:00,4.0
...,...,...
2189995,2021-05-16 17:00:00,45.0
2189996,2021-05-16 17:00:00,46.0
2189997,2021-05-16 17:00:00,47.0
2189998,2021-05-16 17:00:00,48.0


# do_obs

In [8]:
df_do_obs = df_melt['do_obs.csv']

In [9]:
df_do_obs["depth"] = df_do_obs["depth"].astype(float)

# delete the rows which have depth=25
df_do_obs = df_do_obs[df_do_obs["depth"] != 25.0]

In [10]:
valid_depths = list(np.array(range(0, 50)).astype(float))
real_depths = list(np.arange(0, 25, 0.5).astype(float))
depth_dict = dict(zip(real_depths, valid_depths))

df_do_obs["depth"] = df_do_obs["depth"].map(depth_dict)


/tmp/ipykernel_57725/4286374513.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_do_obs["depth"] = df_do_obs["depth"].map(depth_dict)


In [11]:
# get unique depths of the file
unique_depths = df_do_obs['depth'].astype(float).unique()  
sorted_unique_depths = np.sort(unique_depths)

print(sorted_unique_depths)

df_do_obs[df_do_obs['depth'].isna()]
# 1, 33, 35, 37, 39, 41, 

[ 0.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.
 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 34. 36. 38. 40.
 42. 43. 44. 45. 46. 47. 48. 49.]


,datetime,depth,do_obs


In [12]:
df_observed = pd.merge(df_observed, df_do_obs, on=['datetime', 'depth'], how='left')


In [13]:
assert len(df_observed) == 2190000


## Check Correctness

In [14]:
start_time = df_observed['datetime'].min()
end_time = df_observed['datetime'].max()

In [15]:
df_do_obs = df_do_obs[(df_do_obs['datetime'] >= start_time) & (df_do_obs['datetime'] <= end_time)]
df_do_obs

,datetime,depth,do_obs
5,2016-05-22 19:00:00,0.0,11.6
6,2016-06-06 19:00:00,0.0,9.9
7,2016-06-19 19:00:00,0.0,8.3
8,2016-07-04 19:00:00,0.0,9.8
9,2016-07-17 19:00:00,0.0,7.8
...,...,...,...
1395310,2020-05-28 05:00:00,9.0,NaN
1395311,2020-05-28 06:00:00,9.0,NaN
1395312,2020-05-28 07:00:00,9.0,NaN
1395313,2020-05-28 08:00:00,9.0,NaN


In [16]:
# Convert pairs to sets for comparison
df_observed_pairs = set(zip(df_observed['datetime'], df_observed['depth']))
df_do_obs_pairs = set(zip(df_do_obs['datetime'], df_do_obs['depth']))

# Find pairs in df_observed but not in df_do_obs
missing_pairs = df_observed_pairs - df_do_obs_pairs

# Count the number of missing pairs
missing_pairs_count = len(missing_pairs)

print(f"Number of (datetime, depth) pairs in df_observed but not in df_do_obs: {missing_pairs_count}")


Number of (datetime, depth) pairs in df_observed but not in df_do_obs: 1263448


In [17]:
df_do_obs["do_obs"].isna().sum() + missing_pairs_count == df_observed["do_obs"].isna().sum()

True

In [18]:
df_observed

,datetime,depth,do_obs
0,2016-05-17 18:00:00,0.0,NaN
1,2016-05-17 18:00:00,1.0,NaN
2,2016-05-17 18:00:00,2.0,9.964167
3,2016-05-17 18:00:00,3.0,NaN
4,2016-05-17 18:00:00,4.0,NaN
...,...,...,...
2189995,2021-05-16 17:00:00,45.0,NaN
2189996,2021-05-16 17:00:00,46.0,NaN
2189997,2021-05-16 17:00:00,47.0,NaN
2189998,2021-05-16 17:00:00,48.0,NaN


# doc obs

In [19]:
df_doc_obs = df_melt['doc_obs.csv']

In [20]:
df_doc_obs["depth"] = df_doc_obs["depth"].astype(float)
# get unique depths of the file
unique_depths = df_doc_obs['depth'].astype(float).unique()  
sorted_unique_depths = np.sort(unique_depths)

print(sorted_unique_depths)

valid_depths = list(np.array(range(0, 50)).astype(float))
real_depths = list(np.arange(0, 25, 0.5).astype(float))
depth_dict = dict(zip(real_depths, valid_depths))

print(depth_dict)
df_doc_obs["depth"] = df_doc_obs["depth"].map(depth_dict)

[ 0.  1.  2.  3.  4.  5.  6.  8. 10. 12. 14. 15. 16. 18. 19. 20. 22.]
{0.0: 0.0, 0.5: 1.0, 1.0: 2.0, 1.5: 3.0, 2.0: 4.0, 2.5: 5.0, 3.0: 6.0, 3.5: 7.0, 4.0: 8.0, 4.5: 9.0, 5.0: 10.0, 5.5: 11.0, 6.0: 12.0, 6.5: 13.0, 7.0: 14.0, 7.5: 15.0, 8.0: 16.0, 8.5: 17.0, 9.0: 18.0, 9.5: 19.0, 10.0: 20.0, 10.5: 21.0, 11.0: 22.0, 11.5: 23.0, 12.0: 24.0, 12.5: 25.0, 13.0: 26.0, 13.5: 27.0, 14.0: 28.0, 14.5: 29.0, 15.0: 30.0, 15.5: 31.0, 16.0: 32.0, 16.5: 33.0, 17.0: 34.0, 17.5: 35.0, 18.0: 36.0, 18.5: 37.0, 19.0: 38.0, 19.5: 39.0, 20.0: 40.0, 20.5: 41.0, 21.0: 42.0, 21.5: 43.0, 22.0: 44.0, 22.5: 45.0, 23.0: 46.0, 23.5: 47.0, 24.0: 48.0, 24.5: 49.0}


In [21]:
df_observed = pd.merge(df_observed, df_doc_obs, on=['datetime', 'depth'], how='left')
len(df_observed)

2190000

## Check Correctness

In [22]:
df_doc_obs = df_doc_obs[(df_doc_obs['datetime'] >= start_time) & (df_doc_obs['datetime'] <= end_time)]
# Convert pairs to sets for comparison
df_observed_pairs = set(zip(df_observed['datetime'], df_observed['depth']))
df_doc_obs_pairs = set(zip(df_doc_obs['datetime'], df_doc_obs['depth']))

# Find pairs in df_observed but not in df_doc_obs
missing_pairs = df_observed_pairs - df_doc_obs_pairs

# Count the number of missing pairs
missing_pairs_count = len(missing_pairs)

print(f"Number of (datetime, depth) pairs in df_observed but not in df_doc_obs: {missing_pairs_count}")

Number of (datetime, depth) pairs in df_observed but not in df_doc_obs: 2188793


In [23]:
df_doc_obs["doc_obs"].isna().sum() + missing_pairs_count == df_observed["doc_obs"].isna().sum()

True

In [24]:
df_observed["depth"].isna().sum()

0

In [25]:
df_observed[(df_observed["datetime"] == "2016-10-03 19:00:00") & (df_observed["depth"] == 8.0)]

,datetime,depth,do_obs,doc_obs
166858,2016-10-03 19:00:00,8.0,7.5,4.28


In [26]:
df_doc_obs[(df_doc_obs["datetime"] == "2016-10-03 19:00:00") & (df_doc_obs["depth"] == 8.0)]

,datetime,depth,doc_obs
97,2016-10-03 19:00:00,8.0,4.28


# physco_obs

In [27]:
df_physco_obs = df_melt["phyco_obs.csv"]

In [28]:
df_physco_obs = df_physco_obs.drop(columns="depth")
df_physco_obs

,datetime,phyco_obs
0,2019-05-08 23:00:00,0.393333
1,2019-05-09 00:00:00,0.410667
2,2019-05-09 01:00:00,0.448667
3,2019-05-09 02:00:00,0.422333
4,2019-05-09 03:00:00,0.426500
...,...,...
35876,2019-05-08 18:00:00,0.393833
35877,2019-05-08 19:00:00,0.396000
35878,2019-05-08 20:00:00,0.392000
35879,2019-05-08 21:00:00,0.389833


In [29]:
df_observed = pd.merge(df_observed, df_physco_obs, on=['datetime'], how='left')

In [30]:
df_observed

,datetime,depth,do_obs,doc_obs,phyco_obs
0,2016-05-17 18:00:00,0.0,NaN,NaN,371.531667
1,2016-05-17 18:00:00,1.0,NaN,NaN,371.531667
2,2016-05-17 18:00:00,2.0,9.964167,NaN,371.531667
3,2016-05-17 18:00:00,3.0,NaN,NaN,371.531667
4,2016-05-17 18:00:00,4.0,NaN,NaN,371.531667
...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,NaN,NaN,-0.029667
2189996,2021-05-16 17:00:00,46.0,NaN,NaN,-0.029667
2189997,2021-05-16 17:00:00,47.0,NaN,NaN,-0.029667
2189998,2021-05-16 17:00:00,48.0,NaN,NaN,-0.029667


# fdom_obs

In [31]:
df_fdom_obs = df_melt["fdom_obs.csv"]

df_fdom_obs = df_fdom_obs.drop(columns="depth")
df_observed = pd.merge(df_observed, df_fdom_obs, on=['datetime'], how='left')
df_observed

,datetime,depth,do_obs,doc_obs,phyco_obs,fdom_obs
0,2016-05-17 18:00:00,0.0,NaN,NaN,371.531667,NaN
1,2016-05-17 18:00:00,1.0,NaN,NaN,371.531667,NaN
2,2016-05-17 18:00:00,2.0,9.964167,NaN,371.531667,NaN
3,2016-05-17 18:00:00,3.0,NaN,NaN,371.531667,NaN
4,2016-05-17 18:00:00,4.0,NaN,NaN,371.531667,NaN
...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,NaN,NaN,-0.029667,7.682
2189996,2021-05-16 17:00:00,46.0,NaN,NaN,-0.029667,7.682
2189997,2021-05-16 17:00:00,47.0,NaN,NaN,-0.029667,7.682
2189998,2021-05-16 17:00:00,48.0,NaN,NaN,-0.029667,7.682


# poc_obs

In [32]:
df_poc_obs = df_melt["poc_obs.csv"]


In [33]:
df_poc_obs["depth"] = df_poc_obs["depth"].astype(float)
# get unique depths of the file
unique_depths = df_poc_obs['depth'].astype(float).unique()  
sorted_unique_depths = np.sort(unique_depths)

print(sorted_unique_depths)

valid_depths = list(np.array(range(0, 50)).astype(float))
real_depths = list(np.arange(0, 25, 0.5).astype(float))
depth_dict = dict(zip(real_depths, valid_depths))

print(depth_dict)
df_poc_obs["depth"] = df_poc_obs["depth"].map(depth_dict)

[ 3. 10. 12. 14. 20.]
{0.0: 0.0, 0.5: 1.0, 1.0: 2.0, 1.5: 3.0, 2.0: 4.0, 2.5: 5.0, 3.0: 6.0, 3.5: 7.0, 4.0: 8.0, 4.5: 9.0, 5.0: 10.0, 5.5: 11.0, 6.0: 12.0, 6.5: 13.0, 7.0: 14.0, 7.5: 15.0, 8.0: 16.0, 8.5: 17.0, 9.0: 18.0, 9.5: 19.0, 10.0: 20.0, 10.5: 21.0, 11.0: 22.0, 11.5: 23.0, 12.0: 24.0, 12.5: 25.0, 13.0: 26.0, 13.5: 27.0, 14.0: 28.0, 14.5: 29.0, 15.0: 30.0, 15.5: 31.0, 16.0: 32.0, 16.5: 33.0, 17.0: 34.0, 17.5: 35.0, 18.0: 36.0, 18.5: 37.0, 19.0: 38.0, 19.5: 39.0, 20.0: 40.0, 20.5: 41.0, 21.0: 42.0, 21.5: 43.0, 22.0: 44.0, 22.5: 45.0, 23.0: 46.0, 23.5: 47.0, 24.0: 48.0, 24.5: 49.0}


In [34]:

df_observed = pd.merge(df_observed, df_poc_obs, on=['datetime', 'depth'], how='left')
df_observed

,datetime,depth,do_obs,doc_obs,phyco_obs,fdom_obs,poc_obs
0,2016-05-17 18:00:00,0.0,NaN,NaN,371.531667,NaN,NaN
1,2016-05-17 18:00:00,1.0,NaN,NaN,371.531667,NaN,NaN
2,2016-05-17 18:00:00,2.0,9.964167,NaN,371.531667,NaN,NaN
3,2016-05-17 18:00:00,3.0,NaN,NaN,371.531667,NaN,NaN
4,2016-05-17 18:00:00,4.0,NaN,NaN,371.531667,NaN,NaN
...,...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,NaN,NaN,-0.029667,7.682,NaN
2189996,2021-05-16 17:00:00,46.0,NaN,NaN,-0.029667,7.682,NaN
2189997,2021-05-16 17:00:00,47.0,NaN,NaN,-0.029667,7.682,NaN
2189998,2021-05-16 17:00:00,48.0,NaN,NaN,-0.029667,7.682,NaN


## Check Correctness

In [35]:
df_doc_obs_filtered = df_poc_obs[(df_poc_obs['datetime'] >= start_time) & (df_poc_obs['datetime'] <= end_time)]


In [36]:
len(df_doc_obs_filtered)

120

In [37]:
# dataset had one NaN itself
df_observed["poc_obs"].notna().sum()

119

# secchi_obs

In [47]:
df_melt["secchi_obs.csv"]

,datetime,depth,secchi_obs
0,2016-01-26 18:00:00,NA,4.8
1,2016-03-27 19:00:00,NA,1.6
2,2016-04-11 19:00:00,NA,2.1
3,2016-04-24 19:00:00,NA,1.6
4,2016-05-10 19:00:00,NA,5.9
...,...,...,...
109,2022-09-18 19:00:00,NA,1.7
110,2022-10-03 19:00:00,NA,1.6
111,2022-10-19 19:00:00,NA,2.7
112,2022-10-31 19:00:00,NA,2.1


In [38]:
df_secchi_obs = df_melt["secchi_obs.csv"]

df_secchi_obs = df_secchi_obs.drop(columns="depth")

df_observed = pd.merge(df_observed, df_secchi_obs, on=['datetime'], how='left')

In [71]:
df_observed[df_observed["datetime"]=="2016-10-03 19:00:00"]

,datetime,depth,do_obs,doc_obs,phyco_obs,fdom_obs,poc_obs,secchi_obs,temp_obs,scaled_chla_obs
166850,2016-10-03 19:00:00,0.0,7.900000,4.49,1051.94,NaN,NaN,2.4,19.135738,NaN
166851,2016-10-03 19:00:00,1.0,NaN,NaN,1051.94,NaN,NaN,2.4,19.124167,NaN
166852,2016-10-03 19:00:00,2.0,8.690492,NaN,1051.94,NaN,NaN,2.4,19.157603,NaN
166853,2016-10-03 19:00:00,3.0,NaN,NaN,1051.94,NaN,NaN,2.4,18.932500,NaN
166854,2016-10-03 19:00:00,4.0,7.400000,NaN,1051.94,NaN,NaN,2.4,18.829508,NaN
166855,2016-10-03 19:00:00,5.0,NaN,NaN,1051.94,NaN,NaN,2.4,NaN,NaN
166856,2016-10-03 19:00:00,6.0,7.400000,4.56,1051.94,NaN,1.129333,2.4,18.294590,NaN
166857,2016-10-03 19:00:00,7.0,NaN,NaN,1051.94,NaN,NaN,2.4,NaN,NaN
166858,2016-10-03 19:00:00,8.0,7.500000,4.28,1051.94,NaN,NaN,2.4,18.209672,NaN
166859,2016-10-03 19:00:00,9.0,NaN,NaN,1051.94,NaN,NaN,2.4,NaN,NaN


# temp_obs

In [40]:
df_temp_obs = df_melt["temp_obs.csv"]

In [41]:
df_temp_obs["depth"] = df_temp_obs["depth"].astype(float)

# get unique depths of the file
unique_depths = df_temp_obs['depth'].astype(float).unique()  
sorted_unique_depths = np.sort(unique_depths)

print(sorted_unique_depths)

# delete the rows which have depth=25
df_temp_obs = df_temp_obs[df_temp_obs["depth"] != 25.0]

print(depth_dict)
df_temp_obs["depth"] = df_temp_obs["depth"].map(depth_dict)


[ 0.   0.5  1.   1.5  2.   2.5  3.   3.5  4.   4.5  5.   5.5  6.   6.5
  7.   7.5  8.   8.5  9.   9.5 10.  10.5 11.  11.5 12.  12.5 13.  13.5
 14.  14.5 15.  15.5 16.  17.  18.  19.  20.  21.  21.5 22.  22.5 23.
 23.5 24.  24.5 25. ]
{0.0: 0.0, 0.5: 1.0, 1.0: 2.0, 1.5: 3.0, 2.0: 4.0, 2.5: 5.0, 3.0: 6.0, 3.5: 7.0, 4.0: 8.0, 4.5: 9.0, 5.0: 10.0, 5.5: 11.0, 6.0: 12.0, 6.5: 13.0, 7.0: 14.0, 7.5: 15.0, 8.0: 16.0, 8.5: 17.0, 9.0: 18.0, 9.5: 19.0, 10.0: 20.0, 10.5: 21.0, 11.0: 22.0, 11.5: 23.0, 12.0: 24.0, 12.5: 25.0, 13.0: 26.0, 13.5: 27.0, 14.0: 28.0, 14.5: 29.0, 15.0: 30.0, 15.5: 31.0, 16.0: 32.0, 16.5: 33.0, 17.0: 34.0, 17.5: 35.0, 18.0: 36.0, 18.5: 37.0, 19.0: 38.0, 19.5: 39.0, 20.0: 40.0, 20.5: 41.0, 21.0: 42.0, 21.5: 43.0, 22.0: 44.0, 22.5: 45.0, 23.0: 46.0, 23.5: 47.0, 24.0: 48.0, 24.5: 49.0}


/tmp/ipykernel_57725/3192587224.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp_obs["depth"] = df_temp_obs["depth"].map(depth_dict)


In [42]:

df_observed = pd.merge(df_observed, df_temp_obs, on=['datetime', 'depth'], how='left')
df_observed

,datetime,depth,do_obs,doc_obs,phyco_obs,fdom_obs,poc_obs,secchi_obs,temp_obs
0,2016-05-17 18:00:00,0.0,NaN,NaN,371.531667,NaN,NaN,NaN,12.339333
1,2016-05-17 18:00:00,1.0,NaN,NaN,371.531667,NaN,NaN,NaN,12.388167
2,2016-05-17 18:00:00,2.0,9.964167,NaN,371.531667,NaN,NaN,NaN,12.317833
3,2016-05-17 18:00:00,3.0,NaN,NaN,371.531667,NaN,NaN,NaN,12.340500
4,2016-05-17 18:00:00,4.0,NaN,NaN,371.531667,NaN,NaN,NaN,12.329833
...,...,...,...,...,...,...,...,...,...
2189995,2021-05-16 17:00:00,45.0,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN
2189996,2021-05-16 17:00:00,46.0,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN
2189997,2021-05-16 17:00:00,47.0,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN
2189998,2021-05-16 17:00:00,48.0,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN


In [43]:
# df_observed[df_observed["datetime"]=="2016-05-17 18:00:00"]

# Scaled_chla_ob


In [66]:
df_chla_obs = df_melt["scaled_chla_obs.csv"]
df_chla_obs

,datetime,depth,scaled_chla_obs
0,2023-07-10 16:00:00,1.0,0.369355
1,2023-07-10 17:00:00,1.0,0.493347
2,2023-07-10 18:00:00,1.0,0.632831
3,2023-07-10 19:00:00,1.0,0.769164
4,2023-07-10 20:00:00,1.0,0.926300
...,...,...,...
258035,2017-09-06 08:00:00,8.0,1.399767
258036,2018-06-11 08:00:00,8.0,-1.209146
258037,2018-07-09 08:00:00,8.0,NaN
258038,2018-08-06 08:00:00,8.0,0.382004


In [59]:
df_chla_obs["depth"] = df_chla_obs["depth"].astype(float)
df_chla_obs

,datetime,depth,scaled_chla_obs
0,2023-07-10 16:00:00,1.0,0.369355
1,2023-07-10 17:00:00,1.0,0.493347
2,2023-07-10 18:00:00,1.0,0.632831
3,2023-07-10 19:00:00,1.0,0.769164
4,2023-07-10 20:00:00,1.0,0.926300
...,...,...,...
258035,2017-09-06 08:00:00,8.0,1.399767
258036,2018-06-11 08:00:00,8.0,-1.209146
258037,2018-07-09 08:00:00,8.0,NaN
258038,2018-08-06 08:00:00,8.0,0.382004


In [60]:
df_observed = pd.merge(df_observed, df_chla_obs, on=['datetime', 'depth'], how='left')

# Missing values

In [61]:
df_observed.isnull().mean() * 100

datetime            0.000000
depth               0.000000
do_obs             98.943889
doc_obs            99.979635
phyco_obs          41.963529
fdom_obs           75.429038
poc_obs            99.994566
secchi_obs         99.819636
temp_obs           71.142494
scaled_chla_obs    99.532469
dtype: float64

In [65]:
df_observed

,datetime,depth,do_obs,doc_obs,phyco_obs,fdom_obs,poc_obs,secchi_obs,temp_obs,scaled_chla_obs
0,2016-05-17 18:00:00,0.0,NaN,NaN,371.531667,NaN,NaN,NaN,12.339333,NaN
1,2016-05-17 18:00:00,1.0,NaN,NaN,371.531667,NaN,NaN,NaN,12.388167,NaN
2,2016-05-17 18:00:00,2.0,9.964167,NaN,371.531667,NaN,NaN,NaN,12.317833,NaN
3,2016-05-17 18:00:00,3.0,NaN,NaN,371.531667,NaN,NaN,NaN,12.340500,NaN
4,2016-05-17 18:00:00,4.0,NaN,NaN,371.531667,NaN,NaN,NaN,12.329833,NaN
...,...,...,...,...,...,...,...,...,...,...
2190011,2021-05-16 17:00:00,45.0,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN,NaN
2190012,2021-05-16 17:00:00,46.0,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN,NaN
2190013,2021-05-16 17:00:00,47.0,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN,NaN
2190014,2021-05-16 17:00:00,48.0,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN,NaN


# Save

In [62]:
df_main = pd.merge(df_main, df_observed, on=['datetime', 'depth'], how='left')

In [63]:
folder_path = '/raid/sepideh/Project_MCL/1D-AEMpy-UW-metabolism-BM'
output_path = os.path.join(folder_path, "all_data_lake_modeling_mixed.csv")
df_main.to_csv(output_path, index=False)

In [64]:
df_main

,datetime,depth,do_ax01,do_bc02,do_conv05,do_diff04,do_final06,do_initial00,do_pd03,doc_final06,...,secchi_final06,tp_initial,do_obs,doc_obs,phyco_obs,fdom_obs,poc_obs,secchi_obs,temp_obs,scaled_chla_obs
0,2016-05-17 18:00:00,0.0,2.223120e+08,2.230776e+08,2.267030e+08,2.227493e+08,11.377818,2.311300e+08,2.227493e+08,5.943507,...,1.805759,37.097143,NaN,NaN,371.531667,NaN,NaN,NaN,12.339333,NaN
1,2016-05-17 18:00:00,1.0,2.199650e+08,2.204061e+08,2.157519e+08,2.106391e+08,11.377818,2.199650e+08,2.201040e+08,5.943492,...,1.805759,37.097143,NaN,NaN,371.531667,NaN,NaN,NaN,12.388167,NaN
2,2016-05-17 18:00:00,2.0,2.097000e+08,2.099567e+08,2.048007e+08,2.049932e+08,11.377818,2.097000e+08,2.096759e+08,5.943475,...,1.805759,37.097143,9.964167,NaN,371.531667,NaN,NaN,NaN,12.317833,NaN
3,2016-05-17 18:00:00,3.0,2.056250e+08,2.057773e+08,1.991118e+08,2.025911e+08,11.377818,2.056250e+08,2.055109e+08,5.943465,...,1.805759,37.097143,NaN,NaN,371.531667,NaN,NaN,NaN,12.340500,NaN
4,2016-05-17 18:00:00,4.0,2.014500e+08,2.015404e+08,1.990979e+08,1.990979e+08,11.711639,2.014500e+08,2.012874e+08,5.854266,...,1.805759,37.097143,NaN,NaN,371.531667,NaN,NaN,NaN,12.329833,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2190011,2021-05-16 17:00:00,45.0,1.366915e+07,1.366915e+07,1.365258e+07,1.365258e+07,10.707904,1.366915e+07,1.365336e+07,5.108676,...,2.453275,73.383492,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN,NaN
2190012,2021-05-16 17:00:00,46.0,8.411592e+06,8.411592e+06,8.397004e+06,8.397004e+06,10.496256,8.411592e+06,8.398730e+06,5.109180,...,2.453275,73.383492,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN,NaN
2190013,2021-05-16 17:00:00,47.0,4.164276e+06,4.164276e+06,4.149462e+06,4.149462e+06,9.763440,4.164276e+06,4.154155e+06,5.112797,...,2.453275,73.383492,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN,NaN
2190014,2021-05-16 17:00:00,48.0,2.252801e+05,2.252801e+05,2.184360e+05,2.184360e+05,4.368719,2.252801e+05,2.183402e+05,5.145544,...,2.453275,73.383492,NaN,NaN,-0.029667,7.682,NaN,NaN,NaN,NaN
